In [97]:
import math
from typing import Callable

from scipy.stats import chi2
import numpy as np

def est_T(frecuencias: list[int], probabilidades: list[float], n: int, K: int):
    t = 0
    for k in range(K):
        t = t + (frecuencias[k] - n * probabilidades[k]
                 )**2 / (n * probabilidades[k])

    return t

def gen_frecuencias_binomial(probabilidades: list[float], n: int, K: int) -> list[int]:
    n_res = n
    freq = []
    p = probabilidades[0]
    for i in range(K - 1):
        N_i = np.random.binomial(n_res, p)
        freq.append(N_i)

        n_res -= N_i
        p = probabilidades[i + 1] / (1 - sum(probabilidades[:(i + 1)]))

    freq.append(n_res)

    return freq

### Ejercicio 1.
De acuerdo con la teoría genética de Mendel, cierta planta de guisantes debe producir flores blancas, rosas o rojas con probabilidad $1/4$, $1/2$ y $1/4$, respectivamente. Para verificar experimentalmente la teoría, se estudió una muestra de $564$ guisantes, donde se encontró que $141$ produjeron flores blancas, $291$ flores rosas y $132$ flores rojas. Aproximar el *p-valor* de esta muestra:

a. utilizando la prueba de Pearson con aproximación chi-cuadrada,

b. realizando una simulación.

---

### Análisis
Usamos la prueba chi-cuadrada Pearson con el estadístico:
$$
T = \sum_{i=1}^{K}\frac{(N_i - n p_i)^2}{n p_i}
$$

Donde K es la cantidad de valores que puede tener nuestra muestra discreta, se tiene que para cada $i\in \{1,2,\cdots,K\}$:

*   $N_i$ es la frecuencia observada en los datos correspondientes a la clase $i$.
*   $p_i$ es la probabilidad de que la variable aleatoria tome el valor $i$

El *p-valor* ($p$) viene dado por:
$$
p = P(T \geq t_{obs}) \sim P(χ_{k-1} \geq t_{obs})
$$

Si $t_{obs}$ es el estadístico obtenido con la muestra original, el *p-valor* se aproxima mediante simulaciones de la siguiente forma:

$$
p\text{-valor} \approx \frac{\#\{j:t_{(j)}\ge t_{obs}\}}{N_{sim}}.
$$

Siendo $t_{(j)}$ el valor al evaluar T con una muestra generada según F de $H_0$.

---

Nuestro $K$ es igual a $3$ (flores blancas, rosas y rojas) porque la variable $X$ puede tomar $3$ valores distintos.

Ademas:
$$ \text{p-valor} \sim P(χ_{2} \geq t_{obs}) $$

In [94]:
probabilidades = [0.25, 0.50, 0.25]  # probabilidades teóricas
frecuencias = [141, 291, 132]

assert sum(frecuencias) == 564

In [104]:
# Punto a
def punto_a(frecuencias: list[int], probabilidades: list[float]):
    K = len(frecuencias)
    n = sum(frecuencias)

    t_obs = est_T(frecuencias, probabilidades, n, K)
    p_valor = chi2.sf(t_obs, df=(K - 1))

    # print("p-valor: "", 1 - chi2.cdf(t_obs, df=(K - 1)))
    print("p-valor: ", p_valor)

punto_a(frecuencias, probabilidades)

p-valor:  0.6499557054800363


In [105]:
# Punto b
def punto_b(
        frecuencias: list[int],
        probabilidades: list[float],
        gen_freq: Callable[[list[float]], list[float]],
        N_SIM: int = 10_000
    ):

    n = sum(frecuencias)
    K = len(frecuencias)
    t_obs = est_T(frecuencias, probabilidades, n, K)
    
    p_valor = 0
    for _ in range(N_SIM):
        # generar lista de frecuencias utilizando la lista de probabilidades
        freq = gen_freq(probabilidades, n, K)

        # calculo el estadístico para la muestra generada
        t = est_T(freq, probabilidades, n, K)

        if t_obs <= t:
            p_valor += 1

    p_valor = p_valor / N_SIM
    print("p-valor: ", p_valor)

punto_b(frecuencias, probabilidades, gen_frecuencias_binomial)

p-valor:  0.6538


Como *p-valor* no es bajo, no se tiene evidencia suficiente para rechazar $H_0$.